# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

In [ ]:
# Based on the paper's Materials and Methods section, this data represents 17 marketing campaigns.
answer = "17 marketing campaigns"
print(answer)

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [1]:
import pandas as pd

In [10]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [11]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [14]:
# Check shape, dtypes and missing values
print(df.shape)
print("\n")
df.info()
print("\n")

# Check 'unknown' values in categorical columns
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    n_unknown = (df[col] == 'unknown').sum()
    if n_unknown > 0:
        print(f"{col}: {n_unknown} unknowns ({n_unknown/len(df)*100:.1f}%)")

(41188, 21)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   4

The dataset contains 41,188 records and 20 input features and 1 target variable (y). 
There are no explicit NaN values — missing data is encoded as 'unknown' in categorical columns (job, marital, education, default, housing, loan). 
All features are already in their correct data types: numeric columns are int64/float64 and categorical columns are object. 

Note: The feature `duration` will be excluded from modeling as recommended by the dataset authors — call duration is only known after the call ends, making it unavailable at prediction time and unsuitable for a realistic model.

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

**Business Objective:**

The goal is to predict whether a bank client will subscribe to a term deposit (y = 'yes') following a telephone marketing campaign. 

By accurately identifying clients most likely to subscribe, the bank can focus its resources on high-probability targets — reducing the number of calls needed while maintaining a similar number of successful subscriptions. This is especially important given the low success rate (~11%), which means most calls currently result in no conversion.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [15]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Bank client features only (excluding duration and campaign-related features)
bank_features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

# Target
X = df[bank_features]
y = df['y'].map({'yes': 1, 'no': 0})

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nClass balance:\n{y.value_counts(normalize=True).round(3)}")

Features shape: (41188, 7)
Target distribution:
y
0    36548
1     4640
Name: count, dtype: int64

Class balance:
y
0    0.887
1    0.113
Name: proportion, dtype: float64


In [16]:
# Define numeric and categorical columns
numeric_features = ['age']
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan']

# Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
])

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

Following the paper's methodology, we use a 2/3 train / 1/3 test split.  We use `stratify=y` to maintain the class balance in both sets given the  imbalanced nature of the target variable (~11% positive class).

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

print(f"Train set: {X_train.shape[0]} records")
print(f"Test set:  {X_test.shape[0]} records")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest class balance:\n{y_test.value_counts(normalize=True).round(3)}")

Train set: 27595 records
Test set:  13593 records

Train class balance:
y
0    0.887
1    0.113
Name: proportion, dtype: float64

Test class balance:
y
0    0.887
1    0.113
Name: proportion, dtype: float64


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Baseline: always predict the majority class
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_dummy)
print(f"Baseline accuracy (majority class): {baseline_accuracy:.4f}")
print(f"\nThis means any useful model must exceed {baseline_accuracy:.1%} accuracy.")


Baseline accuracy (majority class): 0.8874

This means any useful model must exceed 88.7% accuracy.


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import time

# Build pipeline
logreg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42))
])

# Train
t1 = time.time()
logreg_pipe.fit(X_train, y_train)
logreg_time = time.time() - t1

print(f"Logistic Regression trained in {logreg_time:.4f} seconds")

Logistic Regression trained in 0.1630 seconds


### Problem 9: Score the Model

What is the accuracy of your model?

We evaluate the Logistic Regression model using both accuracy and AUC-ROC.  Given the class imbalance, AUC-ROC is a more meaningful metric than accuracy alone.

In [22]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Predictions
y_pred_logreg = logreg_pipe.predict(X_test)
y_prob_logreg = logreg_pipe.predict_proba(X_test)[:, 1]

# Scores
logreg_train_acc = logreg_pipe.score(X_train, y_train)
logreg_test_acc = accuracy_score(y_test, y_pred_logreg)
logreg_auc = roc_auc_score(y_test, y_prob_logreg)

print(f"Train Accuracy: {logreg_train_acc:.4f}")
print(f"Test Accuracy:  {logreg_test_acc:.4f}")
print(f"AUC-ROC:        {logreg_auc:.4f}")
print(f"\nBaseline accuracy was: {baseline_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_logreg, target_names=['No', 'Yes']))

Train Accuracy: 0.8873
Test Accuracy:  0.8874
AUC-ROC:        0.6544

Baseline accuracy was: 0.8874

Classification Report:
              precision    recall  f1-score   support

          No       0.89      1.00      0.94     12062
         Yes       0.00      0.00      0.00      1531

    accuracy                           0.89     13593
   macro avg       0.44      0.50      0.47     13593
weighted avg       0.79      0.89      0.83     13593



/Users/angel/Workspace/pc_ml_ai/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/angel/Workspace/pc_ml_ai/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/angel/Workspace/pc_ml_ai/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [23]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import pandas as pd
import time

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN':                 KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'SVM':                 SVC(random_state=42)
}

results = []

for name, model in models.items():
    # Build pipeline
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train
    start = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Scores
    train_acc = pipe.score(X_train, y_train)
    test_acc  = pipe.score(X_test, y_test)
    
    # AUC-ROC
    if hasattr(pipe.named_steps['classifier'], 'predict_proba'):
        y_scores = pipe.predict_proba(X_test)[:, 1]
    else:
        y_scores = pipe.decision_function(X_test)
    auc = roc_auc_score(y_test, y_scores)
    
    results.append({
        'Model':          name,
        'Train Time (s)': round(train_time, 4),
        'Train Accuracy': round(train_acc, 4),
        'Test Accuracy':  round(test_acc, 4),
        'AUC-ROC':        round(auc, 4)
    })
    
    print(f"{name} done.")

# Results table
results_df = pd.DataFrame(results).set_index('Model')
results_df

Logistic Regression done.
KNN done.
Decision Tree done.
SVM done.


,Train Time (s),Train Accuracy,Test Accuracy,AUC-ROC
Model,,,,
Logistic Regression,0.1318,0.8873,0.8874,0.6544
KNN,0.0621,0.8897,0.8765,0.5866
Decision Tree,0.3132,0.9196,0.8621,0.5673
SVM,35.3770,0.8881,0.8871,0.5791


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

##### Questions